# IV Ratio Ranking Analysis (5-Minute Bars)

This notebook ranks calendar spread opportunities by implied volatility ratio at entry using **5-minute bars** for more precise timing.

## Key Differences from Hourly Analysis

- **Granularity**: Uses 5-min bars instead of hourly
- **Precision**: More accurate entry/exit timing (3:00pm bar vs 3:00-4:00pm hour)
- **Volume**: Better visibility into when trades actually occurred
- **Spreads**: More accurate high-low ranges for spread estimation

## Hypothesis

Higher IV ratio at entry (short IV / long IV) predicts better P&L:
- Short-term options more "expensive" relative to long-term
- Greater differential IV collapse after earnings
- Better calendar spread profitability


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

from dlt_ibapi.repositories import (
    EarningsCalendarReader,
    OptionBarsReader,
    EquityBarsReader
)
from dlt_ibapi.strategies import (
    filter_tradable_earnings,
    run_batch_calendar_spread_backtest,
    calculate_strategy_stats,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Setup complete")

## Configuration Parameters

Configure the analysis behavior:
- **ANALYSIS_DATE**: The "current" date for filtering (simulates running analysis on this date)
- **TARGET_EARNINGS_DATE**: Optional - analyze only earnings on this specific date
- **INCLUDE_PENDING**: Whether to show unannounced earnings in preview (no P&L computed)

In [11]:
from datetime import datetime, time

# Configuration
ANALYSIS_DATE = date(2025, 11, 17)  # The "current" date for filtering
ANALYSIS_TIME = time(12, 0)  # 3:00 PM = Market close (when we enter trades)
TARGET_EARNINGS_DATE = None  # Set to specific date (e.g., date(2025, 11, 17)) to filter, or None for all
INCLUDE_PENDING = False  # Show unannounced earnings in preview (IV only, no P&L)

# Combine into datetime for comparison
ANALYSIS_DATETIME = datetime.combine(ANALYSIS_DATE, ANALYSIS_TIME)

print(f"📅 Analysis Date/Time: {ANALYSIS_DATETIME.strftime('%Y-%m-%d %H:%M')}")
print(f"🎯 Target Earnings Date: {TARGET_EARNINGS_DATE or 'All dates'}")
print(f"👁️  Include Pending: {INCLUDE_PENDING}")

📅 Analysis Date/Time: 2025-11-17 12:00
🎯 Target Earnings Date: All dates
👁️  Include Pending: False


## Earnings Timing Filter

Filter earnings based on whether they have been announced at the analysis time.

In [12]:
def filter_announced_earnings(earnings_df, analysis_datetime):
    """
    Split earnings into announced (complete) and pending (not announced yet).
    
    Logic:
    - PRE_MARKET: Announced by 9:30 AM on earnings_date
    - AFTER_HOURS: Announced by 4:00 PM on earnings_date
    - UNKNOWN: Treat as AFTER_HOURS (conservative)
    
    Returns:
        (announced_df, pending_df): Two DataFrames
    """
    announced_list = []
    pending_list = []
    
    for _, row in earnings_df.iterrows():
        earnings_date = row['earnings_date']
        earnings_time = row['earnings_time']
        
        # Determine announcement datetime
        if earnings_time == 'PRE_MARKET':
            # Announced before market open (9:30 AM)
            announcement_datetime = datetime.combine(earnings_date, time(9, 30))
        elif earnings_time == 'AFTER_HOURS':
            # Announced after market close (4:00 PM)
            announcement_datetime = datetime.combine(earnings_date, time(16, 0))
        else:  # UNKNOWN
            # Conservative: treat as AFTER_HOURS
            announcement_datetime = datetime.combine(earnings_date, time(16, 0))
        
        # Check if announced by analysis time
        if announcement_datetime <= analysis_datetime:
            announced_list.append(row)
        else:
            pending_list.append(row)
    
    announced_df = pd.DataFrame(announced_list) if announced_list else pd.DataFrame()
    pending_df = pd.DataFrame(pending_list) if pending_list else pd.DataFrame()
    
    return announced_df, pending_df

print("✅ Earnings timing filter function defined")

✅ Earnings timing filter function defined


## 1. Load and Filter Earnings Events

In [13]:
# Load earnings data
earnings_reader = EarningsCalendarReader(database_path='../data_delta', dataset_name='earnings')

# Determine date range for loading
if TARGET_EARNINGS_DATE:
    # Load only the target date
    all_earnings = earnings_reader.get_earnings_on_date(TARGET_EARNINGS_DATE)
    print(f"🎯 Loaded earnings for target date: {TARGET_EARNINGS_DATE}")
else:
    # Load all upcoming earnings
    all_earnings = earnings_reader.get_upcoming_earnings(
        days_ahead=365,
        from_date=date(2025, 1, 1)
    )
    print(f"📊 Total earnings events: {len(all_earnings)}")

# Filter to tradable events (have option data)
option_reader = OptionBarsReader(database_path='../data_delta', dataset_name='options')
tradable_earnings = filter_tradable_earnings(all_earnings, option_reader)
print(f"📈 Tradable earnings (with option data): {len(tradable_earnings)}")

# Split into announced vs pending based on earnings timing
announced_earnings, pending_earnings = filter_announced_earnings(tradable_earnings, ANALYSIS_DATETIME)

print(f"\n{'='*80}")
print(f"EARNINGS STATUS at {ANALYSIS_DATETIME.strftime('%Y-%m-%d %H:%M')}")
print(f"{'='*80}")
print(f"✅ ANNOUNCED (complete, will compute P&L): {len(announced_earnings)} events")
print(f"   Symbols: {', '.join(sorted(announced_earnings['symbol'].unique())) if not announced_earnings.empty else 'None'}")
print(f"\n⏳ PENDING (not announced yet): {len(pending_earnings)} events")
print(f"   Symbols: {', '.join(sorted(pending_earnings['symbol'].unique())) if not pending_earnings.empty else 'None'}")

if len(announced_earnings) == 0:
    print(f"\n⚠️  WARNING: No announced earnings found! Adjust ANALYSIS_DATE or TARGET_EARNINGS_DATE.")

📊 Total earnings events: 43
📈 Tradable earnings (with option data): 29

EARNINGS STATUS at 2025-11-17 12:00
✅ ANNOUNCED (complete, will compute P&L): 16 events
   Symbols: ARBE, ARMK, BRC, HTHT, IGC, JJSF, JKS, NIU, NKLR, NRXP, SOHU, SY, XPEV, YMM, YSG, ZK

⏳ PENDING (not announced yet): 13 events
   Symbols: ACM, DAC, EH, GLAD, HP, IIIV, IMTX, KNDI, LFMD, LU, MGIC, TCOM, XP


## 2. Run Backtest with IV Calculation

This time we enable `calculate_iv=True` to compute implied volatility for each leg.

In [14]:
# Initialize readers
equity_reader = EquityBarsReader(database_path='../data_delta', dataset_name='stocks')

# Run batch backtest WITH IV calculation on ANNOUNCED earnings only
if len(announced_earnings) > 0:
    print(f"{'='*80}")
    print(f"RUNNING BACKTEST ON {len(announced_earnings)} ANNOUNCED EARNINGS")
    print(f"{'='*80}\n")
    
    results_df = run_batch_calendar_spread_backtest(
        earnings_df=announced_earnings,
        option_reader=option_reader,
        equity_reader=equity_reader,
        option_type='C',
        bar_size='5 mins',
        progress=True,
        calculate_iv=True  # Calculate IV metrics
    )
    
    # Mark as announced earnings
    results_df['earnings_status'] = 'ANNOUNCED'
    
    print(f"\n{'='*80}")
    print(f"BACKTEST COMPLETE (ANNOUNCED EARNINGS)")
    print(f"{'='*80}")
    print(f"Successful backtests: {len(results_df)}")
    print(f"Success rate: {len(results_df) / len(announced_earnings) * 100:.1f}%")
else:
    print("⚠️ No announced earnings to backtest")
    results_df = pd.DataFrame()

# Optionally run on pending earnings (IV ranking preview only)
if INCLUDE_PENDING and len(pending_earnings) > 0:
    print(f"\n{'='*80}")
    print(f"RUNNING PREVIEW ON {len(pending_earnings)} PENDING EARNINGS (IV only)")
    print(f"{'='*80}\n")
    
    pending_results_df = run_batch_calendar_spread_backtest(
        earnings_df=pending_earnings,
        option_reader=option_reader,
        equity_reader=equity_reader,
        option_type='C',
        bar_size='5 mins',
        progress=True,
        calculate_iv=True
    )
    
    # Mark as pending and clear P&L (not valid yet)
    pending_results_df['earnings_status'] = 'PENDING'
    pending_results_df['pnl'] = None
    pending_results_df['pnl_pct'] = None
    pending_results_df['pnl_per_contract'] = None
    
    # Combine with announced results
    if not results_df.empty:
        results_df = pd.concat([results_df, pending_results_df], ignore_index=True)
    else:
        results_df = pending_results_df
    
    print(f"\n{'='*80}")
    print(f"PREVIEW COMPLETE (PENDING EARNINGS)")
    print(f"{'='*80}")
    print(f"Pending previews: {len(pending_results_df)}")

RUNNING BACKTEST ON 16 ANNOUNCED EARNINGS


Processing NIU - 2025-11-17 (PRE_MARKET)...
  Spot: $3.89, ATM Strike: $4 → Using $5.0 (best with ≥2 expirations)
  ✓ Calendar spread (Spot: $3.89, Strike: $5.0): Entry=$10.00, P&L=$5.00

Processing ARBE - 2025-11-17 (PRE_MARKET)...
  Spot: $1.45, ATM Strike: $1 ✓
  ⚠️  Backtest failed: Missing entry prices

Processing SY - 2025-11-17 (PRE_MARKET)...
  Spot: $3.67, ATM Strike: $4 → Using $5.0 (best with ≥2 expirations)
  ✓ Calendar spread (Spot: $3.67, Strike: $5.0): Entry=$45.00, P&L=$-45.00

Processing IGC - 2025-11-17 (PRE_MARKET)...
  Spot: $0.36, ATM Strike: $0 → Using $0.5 (best with ≥2 expirations)
  ⚠️  Backtest failed: Degenerate spread (entry_cost=0.0000)

Processing NRXP - 2025-11-17 (PRE_MARKET)...
  Spot: $2.49, ATM Strike: $2 → Using $2.5 (best with ≥2 expirations)
  ✓ Calendar spread (Spot: $2.49, Strike: $2.5): Entry=$70.00, P&L=$20.00

Processing NKLR - 2025-11-17 (PRE_MARKET)...
  Spot: $4.60, ATM Strike: $5 ✓
  ✓ Calendar s

## 3. Filter to Valid IV Data

Only analyze trades where we successfully calculated IV for both legs.

In [15]:
results_df

,symbol,strike,option_type,short_expiry,long_expiry,entry_time,entry_cost,entry_cost_per_contract,pnl,pnl_pct,pnl_per_contract,exit_time,success,failure_reason,iv_short_entry,iv_long_entry,iv_ratio_entry,earnings_status
0,NIU,5.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.10,10.0,5.000000e-02,5.000000e+01,5.000000e+00,2025-11-17 16:00:00,True,None,1.768207,0.993188,1.780335,ANNOUNCED
1,SY,5.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.45,45.0,-4.500000e-01,-1.000000e+02,-4.500000e+01,2025-11-17 16:00:00,True,None,2.557925,2.134814,1.198195,ANNOUNCED
2,NRXP,2.5,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.70,70.0,2.000000e-01,2.857143e+01,2.000000e+01,2025-11-17 16:00:00,True,None,2.621713,3.501955,0.748643,ANNOUNCED
3,NKLR,5.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.35,35.0,5.000000e-02,1.428571e+01,5.000000e+00,2025-11-17 16:00:00,True,None,3.052841,1.812183,1.684620,ANNOUNCED
4,SOHU,15.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.10,10.0,-5.551115e-17,-5.551115e-14,-5.551115e-15,2025-11-17 16:00:00,True,None,0.919756,0.407744,2.255718,ANNOUNCED
5,XPEV,25.0,C,2025-11-21,2025-11-28,2025-11-16 15:00:00,0.24,24.0,-1.000000e-02,-4.166667e+00,-1.000000e+00,2025-11-17 16:00:00,True,None,0.982755,0.761040,1.291331,ANNOUNCED
6,HTHT,45.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,1.15,115.0,-5.000000e-02,-4.347826e+00,-5.000000e+00,2025-11-17 16:00:00,True,None,0.628386,0.455005,1.381053,ANNOUNCED
7,YMM,12.5,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.50,50.0,-4.500000e-01,-9.000000e+01,-4.500000e+01,2025-11-17 16:00:00,True,None,0.830482,0.647743,1.282117,ANNOUNCED
8,ARMK,38.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,0.25,25.0,1.200000e+00,4.800000e+02,1.200000e+02,2025-11-17 16:00:00,True,None,0.829795,0.362247,2.290689,ANNOUNCED
9,BRC,75.0,C,2025-11-21,2025-12-19,2025-11-16 15:00:00,1.60,160.0,0.000000e+00,0.000000e+00,0.000000e+00,2025-11-17 16:00:00,True,None,0.753309,0.456357,1.650701,ANNOUNCED


In [16]:
# Filter to trades with valid IV data from ANNOUNCED earnings only
announced_only_df = results_df[results_df['earnings_status'] == 'ANNOUNCED'].copy()

valid_iv_df = announced_only_df[
    announced_only_df['iv_short_entry'].notna() & 
    announced_only_df['iv_long_entry'].notna() &
    announced_only_df['iv_ratio_entry'].notna()
].copy()

print(f"{'='*80}")
print(f"IV DATA FILTERING (ANNOUNCED EARNINGS ONLY)")
print(f"{'='*80}")
print(f"Announced results: {len(announced_only_df)}")
print(f"Valid IV data: {len(valid_iv_df)} / {len(announced_only_df)}")

# Check for pending earnings
pending_count = len(results_df[results_df['earnings_status'] == 'PENDING'])
if pending_count > 0:
    print(f"\n⏳ Excluded {pending_count} pending earnings from P&L analysis (earnings not announced yet)")

if len(valid_iv_df) > 0:
    print(f"\nIV Ratio Statistics (Announced Earnings):")
    print(f"  Mean: {valid_iv_df['iv_ratio_entry'].mean():.3f}")
    print(f"  Median: {valid_iv_df['iv_ratio_entry'].median():.3f}")
    print(f"  Min: {valid_iv_df['iv_ratio_entry'].min():.3f}")
    print(f"  Max: {valid_iv_df['iv_ratio_entry'].max():.3f}")
    print(f"\nSample trades:")
    print(valid_iv_df[['symbol', 'iv_short_entry', 'iv_long_entry', 'iv_ratio_entry', 'pnl_per_contract']].head(10))
else:
    print("⚠️ No trades with valid IV data from announced earnings")

IV DATA FILTERING (ANNOUNCED EARNINGS ONLY)
Announced results: 13
Valid IV data: 13 / 13

IV Ratio Statistics (Announced Earnings):
  Mean: 1.613
  Median: 1.651
  Min: 0.749
  Max: 2.291

Sample trades:
  symbol  iv_short_entry  iv_long_entry  iv_ratio_entry  pnl_per_contract
0    NIU        1.768207       0.993188        1.780335      5.000000e+00
1     SY        2.557925       2.134814        1.198195     -4.500000e+01
2   NRXP        2.621713       3.501955        0.748643      2.000000e+01
3   NKLR        3.052841       1.812183        1.684620      5.000000e+00
4   SOHU        0.919756       0.407744        2.255718     -5.551115e-15
5   XPEV        0.982755       0.761040        1.291331     -1.000000e+00
6   HTHT        0.628386       0.455005        1.381053     -5.000000e+00
7    YMM        0.830482       0.647743        1.282117     -4.500000e+01
8   ARMK        0.829795       0.362247        2.290689      1.200000e+02
9    BRC        0.753309       0.456357        1.650701 

## 3.5. Earnings Status Summary

Overview of which earnings were included vs excluded based on timing.

In [17]:
if not tradable_earnings.empty:
    # Create summary with timing details
    summary_data = []
    
    for _, row in tradable_earnings.iterrows():
        symbol = row['symbol']
        earnings_date = row['earnings_date']
        earnings_time = row['earnings_time']
        
        # Determine announcement datetime
        if earnings_time == 'PRE_MARKET':
            announcement_time = '9:30 AM'
            announcement_datetime = datetime.combine(earnings_date, time(9, 30))
        elif earnings_time == 'AFTER_HOURS':
            announcement_time = '4:00 PM'
            announcement_datetime = datetime.combine(earnings_date, time(16, 0))
        else:  # UNKNOWN
            announcement_time = '4:00 PM (assumed)'
            announcement_datetime = datetime.combine(earnings_date, time(16, 0))
        
        # Status at analysis time
        is_announced = announcement_datetime <= ANALYSIS_DATETIME
        status = '✅ ANNOUNCED' if is_announced else '⏳ PENDING'
        
        # Check if successful backtest
        in_results = symbol in results_df['symbol'].values if not results_df.empty else False
        
        summary_data.append({
            'Symbol': symbol,
            'Earnings Date': earnings_date,
            'Timing': earnings_time,
            'Announcement': f"{earnings_date} {announcement_time}",
            'Status': status,
            'In Analysis': '✓' if (is_announced and in_results) else '✗'
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Display table
    print(f"{'='*80}")
    print(f"EARNINGS STATUS BREAKDOWN")
    print(f"{'='*80}\n")
    
    # Group by status
    announced_symbols = summary_df[summary_df['Status'] == '✅ ANNOUNCED']
    pending_symbols = summary_df[summary_df['Status'] == '⏳ PENDING']
    
    print(f"✅ ANNOUNCED Earnings ({len(announced_symbols)}):")
    if not announced_symbols.empty:
        print(announced_symbols[['Symbol', 'Earnings Date', 'Timing', 'In Analysis']].to_string(index=False))
    
    print(f"\n⏳ PENDING Earnings ({len(pending_symbols)}):")
    if not pending_symbols.empty:
        print(pending_symbols[['Symbol', 'Earnings Date', 'Timing', 'Announcement']].to_string(index=False))
        print(f"\n   → These earnings have NOT been announced yet at {ANALYSIS_DATETIME.strftime('%Y-%m-%d %H:%M')}")
        print(f"   → P&L cannot be computed (earnings impact not yet reflected in option prices)")
    else:
        print("   (None)")
else:
    print("⚠️ No tradable earnings to summarize")

EARNINGS STATUS BREAKDOWN

✅ ANNOUNCED Earnings (16):
Symbol Earnings Date     Timing In Analysis
   NIU    2025-11-17 PRE_MARKET           ✓
  ARBE    2025-11-17 PRE_MARKET           ✗
    SY    2025-11-17 PRE_MARKET           ✓
   IGC    2025-11-17 PRE_MARKET           ✗
  NRXP    2025-11-17 PRE_MARKET           ✓
  NKLR    2025-11-17 PRE_MARKET           ✓
  SOHU    2025-11-17 PRE_MARKET           ✓
  XPEV    2025-11-17 PRE_MARKET           ✓
  HTHT    2025-11-17 PRE_MARKET           ✓
   YMM    2025-11-17 PRE_MARKET           ✓
  ARMK    2025-11-17 PRE_MARKET           ✓
    ZK    2025-11-17 PRE_MARKET           ✗
   BRC    2025-11-17 PRE_MARKET           ✓
  JJSF    2025-11-17 PRE_MARKET           ✓
   JKS    2025-11-17 PRE_MARKET           ✓
   YSG    2025-11-17 PRE_MARKET           ✓

⏳ PENDING Earnings (13):
Symbol Earnings Date      Timing                 Announcement
  TCOM    2025-11-17 AFTER_HOURS           2025-11-17 4:00 PM
  LFMD    2025-11-17 AFTER_HOURS           2025-

## 4. Correlation Analysis: IV Ratio vs P&L (Announced Earnings Only)

Test the hypothesis: Does higher IV ratio predict better P&L?

**Note**: Only analyzing announced earnings where P&L is valid.

In [ ]:
if len(valid_iv_df) > 0:
    correlation = valid_iv_df['iv_ratio_entry'].corr(valid_iv_df['pnl_per_contract'])
    
    print("="*80)
    print("CORRELATION ANALYSIS: IV Ratio vs P&L (Announced Earnings)")
    print("="*80)
    print(f"Pearson correlation: {correlation:.3f}")
    print(f"Sample size: {len(valid_iv_df)} announced earnings")
    
    if correlation > 0.3:
        print("✅ Strong positive correlation: Higher IV ratio → Better P&L")
    elif correlation > 0.1:
        print("⚠️ Weak positive correlation")
    elif correlation < -0.1:
        print("❌ Negative correlation: Higher IV ratio → Worse P&L")
    else:
        print("❓ No significant correlation")
    
    # Scatter plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(valid_iv_df['iv_ratio_entry'], valid_iv_df['pnl_per_contract'], alpha=0.6)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1, label='Breakeven')
    
    # Add trend line
    z = np.polyfit(valid_iv_df['iv_ratio_entry'], valid_iv_df['pnl_per_contract'], 1)
    p = np.poly1d(z)
    ax.plot(valid_iv_df['iv_ratio_entry'], p(valid_iv_df['iv_ratio_entry']), 
            "r-", linewidth=2, label=f'Trend (r={correlation:.3f})')
    
    ax.set_xlabel('IV Ratio at Entry (Short IV / Long IV)', fontsize=12, fontweight='bold')
    ax.set_ylabel('P&L per Contract ($)', fontsize=12, fontweight='bold')
    ax.set_title(f'Calendar Spread P&L vs Pre-Earnings IV Ratio\n(Announced Earnings Only, n={len(valid_iv_df)})', 
                 fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data for correlation analysis")

## 5. Rank Opportunities by IV Ratio (Announced Earnings Only)

Divide trades into quartiles by IV ratio and compare performance.

In [ ]:
if len(valid_iv_df) > 0:
    # Create quartiles
    valid_iv_df['iv_ratio_quartile'] = pd.qcut(
        valid_iv_df['iv_ratio_entry'], 
        q=4, 
        labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
    )
    
    # Analyze by quartile
    quartile_stats = valid_iv_df.groupby('iv_ratio_quartile').agg({
        'pnl_per_contract': ['count', 'mean', 'median', 'std'],
        'iv_ratio_entry': ['min', 'max'],
    }).round(2)
    
    quartile_stats.columns = ['Count', 'Mean P&L', 'Median P&L', 'Std Dev', 'IV Ratio Min', 'IV Ratio Max']
    
    print("="*80)
    print("PERFORMANCE BY IV RATIO QUARTILE (Announced Earnings)")
    print("="*80)
    print(f"Sample size: {len(valid_iv_df)} trades\n")
    print(quartile_stats)
    
    # Win rate by quartile
    win_rates = valid_iv_df.groupby('iv_ratio_quartile').apply(
        lambda x: (x['pnl_per_contract'] > 0).sum() / len(x) * 100
    )
    print(f"\nWin Rate by Quartile:")
    for q, wr in win_rates.items():
        print(f"  {q}: {wr:.1f}%")
    
    # Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Mean P&L by quartile
    ax1 = axes[0]
    quartile_stats['Mean P&L'].plot(kind='bar', ax=ax1, color='steelblue', alpha=0.7)
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax1.set_xlabel('IV Ratio Quartile', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Mean P&L per Contract ($)', fontsize=11, fontweight='bold')
    ax1.set_title(f'Average P&L by IV Ratio Quartile\n(Announced Earnings, n={len(valid_iv_df)})', 
                  fontsize=12, fontweight='bold')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Win rate by quartile
    ax2 = axes[1]
    win_rates.plot(kind='bar', ax=ax2, color='green', alpha=0.7)
    ax2.set_xlabel('IV Ratio Quartile', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Win Rate (%)', fontsize=11, fontweight='bold')
    ax2.set_title(f'Win Rate by IV Ratio Quartile\n(Announced Earnings, n={len(valid_iv_df)})', 
                  fontsize=12, fontweight='bold')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data for quartile analysis")

## 6. Compare Top vs Bottom Quartile (Announced Earnings Only)

Direct comparison: Best opportunities (highest IV ratio) vs worst (lowest IV ratio).

In [9]:
if len(valid_iv_df) >= 8:  # Need at least 8 trades for quartiles
    top_quartile = valid_iv_df[valid_iv_df['iv_ratio_quartile'] == 'Q4 (High)']
    bottom_quartile = valid_iv_df[valid_iv_df['iv_ratio_quartile'] == 'Q1 (Low)']
    
    print("="*80)
    print("TOP QUARTILE (Highest IV Ratio) vs BOTTOM QUARTILE (Lowest IV Ratio)")
    print("="*80)
    
    print(f"\nTOP QUARTILE (Q4):")
    print(f"  Count: {len(top_quartile)}")
    print(f"  IV Ratio Range: {top_quartile['iv_ratio_entry'].min():.3f} - {top_quartile['iv_ratio_entry'].max():.3f}")
    print(f"  Mean P&L: ${top_quartile['pnl_per_contract'].mean():.2f}")
    print(f"  Win Rate: {(top_quartile['pnl_per_contract'] > 0).sum() / len(top_quartile) * 100:.1f}%")
    
    print(f"\nBOTTOM QUARTILE (Q1):")
    print(f"  Count: {len(bottom_quartile)}")
    print(f"  IV Ratio Range: {bottom_quartile['iv_ratio_entry'].min():.3f} - {bottom_quartile['iv_ratio_entry'].max():.3f}")
    print(f"  Mean P&L: ${bottom_quartile['pnl_per_contract'].mean():.2f}")
    print(f"  Win Rate: {(bottom_quartile['pnl_per_contract'] > 0).sum() / len(bottom_quartile) * 100:.1f}%")
    
    diff_pnl = top_quartile['pnl_per_contract'].mean() - bottom_quartile['pnl_per_contract'].mean()
    print(f"\n💰 DIFFERENCE: ${diff_pnl:.2f} per contract")
    
    if diff_pnl > 50:
        print("✅ STRONG SIGNAL: High IV ratio significantly outperforms low IV ratio")
    elif diff_pnl > 20:
        print("✅ MODERATE SIGNAL: High IV ratio tends to outperform")
    elif diff_pnl < -20:
        print("❌ INVERSE SIGNAL: Low IV ratio performs better (hypothesis rejected)")
    else:
        print("⚠️ WEAK SIGNAL: No significant difference")
else:
    print("⚠️ Insufficient data for quartile comparison (need at least 8 trades)")

TOP QUARTILE (Highest IV Ratio) vs BOTTOM QUARTILE (Lowest IV Ratio)

TOP QUARTILE (Q4):
  Count: 5
  IV Ratio Range: 2.001 - 2.291
  Mean P&L: $24.00
  Win Rate: 20.0%

BOTTOM QUARTILE (Q1):
  Count: 6
  IV Ratio Range: 0.749 - 1.291
  Mean P&L: $0.83
  Win Rate: 16.7%

💰 DIFFERENCE: $23.17 per contract
✅ MODERATE SIGNAL: High IV ratio tends to outperform


## 7. Inspect Best and Worst Trades (Announced Earnings Only)

Look at individual trades ranked by IV ratio.

In [ ]:
if len(valid_iv_df) > 0:
    # Sort by IV ratio
    sorted_df = valid_iv_df.sort_values('iv_ratio_entry', ascending=False)
    
    print("="*80)
    print("TOP 10 OPPORTUNITIES (Highest IV Ratio)")
    print("="*80)
    print(sorted_df[[
        'symbol', 'iv_ratio_entry', 'iv_short_entry', 'iv_long_entry', 
        'entry_cost_per_contract', 'pnl_per_contract', 'pnl_pct'
    ]].head(10).to_string(index=False))
    
    print(f"\n{'='*80}")
    print("BOTTOM 10 OPPORTUNITIES (Lowest IV Ratio)")
    print("="*80)
    print(sorted_df[[
        'symbol', 'iv_ratio_entry', 'iv_short_entry', 'iv_long_entry', 
        'entry_cost_per_contract', 'pnl_per_contract', 'pnl_pct'
    ]].tail(10).to_string(index=False))
else:
    print("⚠️ No trades to inspect")

## 8. Save Results (Announced Earnings Only)

In [ ]:
if len(valid_iv_df) > 0:
    # Save with timestamp
    filename = f'iv_ratio_ranking_results_{ANALYSIS_DATE.strftime("%Y%m%d")}.csv'
    valid_iv_df.to_csv(filename, index=False)
    print(f"✅ Results saved to {filename} ({len(valid_iv_df)} trades)")
    
    # Summary stats
    print(f"\n{'='*80}")
    print(f"SUMMARY (Announced Earnings Only)")
    print(f"{'='*80}")
    print(f"Analysis date: {ANALYSIS_DATETIME.strftime('%Y-%m-%d %H:%M')}")
    print(f"Total trades analyzed: {len(valid_iv_df)}")
    print(f"Mean IV ratio: {valid_iv_df['iv_ratio_entry'].mean():.3f}")
    print(f"Mean P&L: ${valid_iv_df['pnl_per_contract'].mean():.2f}")
    print(f"Correlation (IV ratio vs P&L): {valid_iv_df['iv_ratio_entry'].corr(valid_iv_df['pnl_per_contract']):.3f}")
    
    # Pending earnings note
    if len(pending_earnings) > 0:
        print(f"\n⏳ Note: {len(pending_earnings)} pending earnings excluded (not announced yet)")
else:
    print("⚠️ No results to save")

## 9. Earnings Timing Logic (Documentation)

This notebook is **earnings-time aware**, meaning it only computes P&L for earnings that have been announced by the analysis time.

### Why This Matters

Calendar spreads profit from IV collapse after earnings. If we analyze trades before earnings are announced:
- ❌ **Entry prices**: Valid (captured at 3:00 PM before announcement)  
- ❌ **Exit prices**: Invalid (not yet affected by earnings announcement)
- ❌ **P&L**: Invalid (earnings impact not yet priced in)

### Filtering Logic

The notebook uses `ANALYSIS_DATE` and `ANALYSIS_TIME` (default: 3:00 PM) to determine which earnings have been announced:

| Earnings Timing | Announcement Time | Example | Status at 3:00 PM |
|-----------------|-------------------|---------|-------------------|
| **PRE_MARKET** | Before 9:30 AM on earnings_date | ARMK on 2025-11-17 | ✅ Announced (include in P&L) |
| **AFTER_HOURS** | After 4:00 PM on earnings_date | TCOM on 2025-11-17 | ⏳ Pending (exclude from P&L) |
| **UNKNOWN** | Assume 4:00 PM (conservative) | EH on 2025-11-17 | ⏳ Pending (exclude from P&L) |

### Configuration Parameters

Adjust these cells at the top of the notebook:

```python
ANALYSIS_DATE = date(2025, 11, 17)  # The "current" date for filtering
ANALYSIS_TIME = time(15, 0)         # 3:00 PM = Market close (entry time)
TARGET_EARNINGS_DATE = None         # Filter to specific date, or None for all
INCLUDE_PENDING = False             # Show unannounced earnings (IV only, no P&L)
```

### Historical Analysis

To analyze a specific past earnings date:

```python
# Analyze earnings from Nov 13, 2025 (simulate running analysis on that day)
ANALYSIS_DATE = date(2025, 11, 13)
TARGET_EARNINGS_DATE = date(2025, 11, 13)  # Optional: only show this date
```

### Preview Mode

To see IV rankings for pending earnings (without P&L):

```python
INCLUDE_PENDING = True  # Shows both announced (with P&L) and pending (IV only)
```

This allows you to rank opportunities before earnings are announced, but P&L will be `None` until after the announcement.

## 10. Conclusions

**Key Questions**:
1. Is there a correlation between pre-earnings IV ratio and P&L?
2. Does ranking by IV ratio help select better opportunities?
3. What threshold IV ratio should we use for trade selection?

**Important Notes**:
- Analysis only includes **announced earnings** (where P&L is valid)
- Pending earnings (not announced yet) are excluded from P&L correlation
- Use `TARGET_EARNINGS_DATE` parameter to analyze specific historical dates
- Use `INCLUDE_PENDING=True` to preview IV rankings for upcoming earnings

**Next Steps**:
- If correlation is strong: Use IV ratio as a filter in live trading
- If correlation is weak: Explore other signals (absolute IV level, DTE, moneyness, etc.)
- Combine with other notebooks (03 for single-trade analysis, 04 for batch backtest, 05 for vol surfaces)